# The scenarios with sliders

The shocks of [Five scenarios](chile_scenarios.ipynb) as sliders: move one or more sliders, give
the scenario a name and press **Run scenario**. The solution is kept and shown next to the
baseline and the scenarios run before it.

:::{note}
The sliders only work in a running notebook. The book shows a picture of the widget as it opens.
To use them here, press the power button (*start compute environment*) at the top of the page
and run the cells in order: Python starts in your browser and installs ModelFlow, which takes a
minute the first time. In [JupyterLite](https://ibhansen.github.io/chile_lite/) the code can also
be changed.
:::

| Tab | Slider | Variable | How the value is used | Value in Five scenarios |
|---|---|---|---|---|
| VAT | VAT rate, change | `TAU_CONS` | added | 0.02 |
| Pensions | Pension spending, permanent step | `G_P_STEP` | added in the first year | 0.005 |
| Health | Elasticity to the 80+ share | `HEALTH_EL` | set | 0.50 (baseline 0.66) |
| Female participation | Participation rate, change | `P_F_1559` | added | 0.08 |
| Copper | Steady-state copper price, % change | `PCOPPER_SS` | changed by percent | −20 |

All sliders act on the same years, 2025–2070, so a shock starts in 2025. In
[Five scenarios](chile_scenarios.ipynb) some shocks start later, and female participation rises
gradually, so the results differ slightly; [Testing the scenario widget](chile_widget_test.ipynb)
shows by how much.

:::{dropdown} The slider definitions, `chile_widgets.py`
```{literalinclude} chile_widgets.py
:language: python
```
:::


In [ ]:
import sys

if sys.platform == 'emscripten':  # in the browser: JupyterLite or the book's live code (thebe-lite)
    import micropip
    await micropip.install('modelflowib>=2.87')
    # the book's live code has no files, so the model and the slider definitions are fetched
    # from the chile_lite repo (JupyterLite has them already)
    from pathlib import Path
    from pyodide.http import pyfetch
    for file in ['data/chilepens.pcim', 'chile_widgets.py']:
        # the browser keeps files between reloads, also a 404 page saved by a failed download
        if not Path(file).exists() or Path(file).read_bytes().startswith(b'404'):
            url = f'https://raw.githubusercontent.com/IbHansen/chile_lite/main/content/{file}'
            response = await pyfetch(url)
            if not response.ok:
                raise RuntimeError(f'Could not download {url}: {response.status} {response.status_text}')
            Path(file).parent.mkdir(exist_ok=True)
            Path(file).write_bytes(await response.bytes())
    sys.path.insert(0, '')        # the notebook's folder, for chile_widgets.py

In [13]:
from modelclass import model
from chile_widgets import scenario_widget
import pandas as pd
%matplotlib inline

model.widescreen()
model.scroll_off()

<IPython.core.display.Javascript object>

## Load the model

Loading solves the model over 2025–2070, the years the sliders act on.

In [19]:
mchile, bline = model.modelload('data/chilepens', run=True, keep='Baseline')

Zipped file read:  data\chilepens.pcim


## The widget

- **Run scenario** solves the model with the slider values and keeps the solution under the
  name in *Scenario name*.
- **Reset to start** moves all sliders back to no change.
- **Use as baseline** makes the last scenario the one later scenarios are compared with.
- *Display variables* chooses which variables the viewer below offers.

In [16]:
scenario_widget(mchile)

:::{figure} graph/scenario_widget.png
:alt: The scenario widget: tabs with the sliders, the buttons, the variable list and a chart of the baseline debt
:width: 100%

The widget as it opens: the VAT tab, the buttons, and the viewer showing gross government debt in
the baseline.
:::